# Optimiser en Julia avec JuMP
**Journée Julia pour les statistiques et science des données — Groupe Calcul CNRS**  
Xavier Gandibleux — Nantes Université — 15 Juin 2026

---
## Installation des packages

À faire **une seule fois** pour ajouter les packages à la distribution Julia.

In [ ]:
using Pkg

# Manière directe et basique :

Pkg.add("JuMP")
Pkg.add("GLPK")
Pkg.add("MultiObjectiveAlgorithms")

Pkg.add("Plots")
Pkg.add("LaTeXStrings")


# Manière propre qui déclenche une installation uniquement si package est absent :

#=
using Pkg
packages = ["JuMP", "GLPK", "MultiObjectiveAlgorithms", "Plots", "LaTeXStrings"]
installes = [p.name for p in values(Pkg.dependencies())]
a_installer = filter(p -> p ∉ installes, packages)
if !isempty(a_installer)
    Pkg.add(a_installer)
end
=#

---
## Partie 1 — Modélisation explicite avec JuMP

### Le problème d'optimisation

$$
\begin{array}{llrcrrl}
\max        & z(x) = & x_1 & +     & 3x_2 &        &      \\
\text{s.t.} &        & x_1 & +     &  x_2 & \leq 14 & \qquad (1) \\
            &        &-2x_1 & +    & 3x_2 & \leq 12 & \qquad (2) \\
            &        & 2x_1 & -    &  x_2 & \leq 12 & \qquad (3) \\
            &        & x_1  & ,\, & x_2  & \geq 0  & \qquad (4)
\end{array}
$$

### Créer un modèle

Syntaxe : `model = Model(solver)`

In [ ]:
using JuMP, GLPK

modLP = Model(GLPK.Optimizer)

### Définir les variables

Syntaxe : `@variable(model, variable)`  
Par défaut, les variables sont **continues** et **non bornées**.

In [ ]:
@variable(modLP, x1 >= 0)
@variable(modLP, x2 >= 0)

### Définir la fonction objectif

Syntaxe : `@objective(model, sense, expression)`

In [ ]:
@objective(modLP, Max, x1 + 3x2)

### Définir les contraintes

Syntaxe : `@constraint(model, id, constraint)`

In [ ]:
@constraint(modLP, cst1, x1 + x2 <= 14)
@constraint(modLP, cst2, -2x1 + 3x2 <= 12)
@constraint(modLP, cst3, 2x1 - x2 <= 12)

### Résoudre le problème

Syntaxe : `optimize!(model)`

In [ ]:
optimize!(modLP)

### Extraire les résultats

In [ ]:
@show termination_status(modLP)  # ou aussi : is_solved_and_feasible(modLP)
@show solution_summary(modLP)

@show objective_value(modLP)
@show value(x1), value(x2)

### Complément : types de variables disponibles

```julia
@variable(model, x)            # x libre (non borné)
@variable(model, x >= lb)      # x borné inférieurement
@variable(model, x <= ub)      # x borné supérieurement
@variable(model, lb <= x <= ub)  # x doublement borné
@variable(model, x == 2)       # x fixé
@variable(model, x >= 0, Int)  # x entier
@variable(model, x, Bin)       # x binaire {0,1}
```

---
## Partie 2 — Modélisation implicite avec JuMP

La modélisation implicite permet de décrire un modèle à partir de **données** (vecteurs, matrices) plutôt que d'écrire chaque contrainte explicitement.

### Modélisation implicite — variables et objectif

Syntaxe pour un vecteur de variables : `@variable(model, x[1:n] >= 0)`  
Syntaxe pour l'objectif vectoriel : `@objective(model, Max, c' * x)`

#### Données du problème

In [ ]:
c = [1, 3]             # coefficients objectif
T = [1 1; -2 3; 2 -1]  # matrice des contraintes
d = [14, 12, 12]       # second membre

#### Modèle JuMP

In [ ]:
# using JuMP, GLPK

m = length(c)       # nombre de variables
n = length(d)       # nombre de contraintes

modLP2 = Model(GLPK.Optimizer)

@variable(modLP2, x[1:m] ≥ 0)
@objective(modLP2, Max, sum(c[j]*x[j] for j=1:m))  # ou aussi :  @objective(modLP2, Max, c' * x)
@constraint(modLP2, cst[i=1:n], sum(T[i,j] * x[j] for j in 1:m) ≤ d[i])

optimize!(modLP2)

@show termination_status(modLP2)
@show objective_value(modLP2)
@show value.(x)

---
## Partie 3 — Optimisation multi-objectif avec MultiObjectiveAlgorithms

### Le problème bi-objectif

$$
\begin{array}{llrcrrl}
\max & z_1 = & x_1  & +  & x_2  &           &     \\
\min & z_2 = & x_1  & +  & 3x_2 &           &     \\
\text{s.t.} && 2x_1 & +  & 3x_2 & \leq 30   & \qquad (1) \\
            && 3x_1 & +  & 2x_2 & \leq 30   & \qquad(2) \\
            &&  x_1 & -  &  x_2 & \leq 5.5  & \qquad(3) \\
            &&  x_1 & ,\,&  x_2 & \in \mathbb{N} & \qquad (4)
\end{array}
$$

### Données

In [ ]:
c1 = [1, 1]
c2 = [1, 3]
A  = [2 3; 3 2; 1 -1]
b  = [30, 30, 5.5]

### Codage du problème MOO avec JuMP et MultiObjectiveAlgorithms

In [ ]:
using JuMP, GLPK
import MultiObjectiveAlgorithms as MOA

model = Model()

@variable(model, x1 >= 0, Int)
@variable(model, x2 >= 0, Int)

@expression(model, fct1, x1 + x2)       # à maximiser
@expression(model, fct2, x1 + 3 * x2)   # à minimiser
@objective(model, Max, [fct1, (-1) * fct2])

@constraint(model, 2*x1 + 3*x2 ≤ 30)
@constraint(model, 3*x1 + 2*x2 ≤ 30)
@constraint(model,   x1 -   x2 ≤ 5.5)

### Sélection du solveur et de l'algorithme

In [ ]:
# Solveur MIP
set_optimizer(model, ()->MOA.Optimizer(GLPK.Optimizer))

# Algorithme epsilon-constraint (step=1 par défaut)
set_attribute(model, MOA.Algorithm(), MOA.EpsilonConstraint())

# Résolution
#set_silent(model)
optimize!(model)

### Extraction de $X_E$ et $Y_N$

In [ ]:
for i in 1:result_count(model)
    z1_opt = objective_value(model; result = i)[1]
    z2_opt = -1 * objective_value(model; result = i)[2]
    x1_opt = value(x1; result = i)
    x2_opt = value(x2; result = i)
    println("Solution $i : z1=$(z1_opt)  z2=$(z2_opt)  |  x1=$(x1_opt)  x2=$(x2_opt)")
end

### Affichage graphique de $Y_N$

In [ ]:
using Plots
using LaTeXStrings

Z1_opt = []; Z2_opt = []
for i in 1:result_count(model)
    push!(Z1_opt, objective_value(model; result = i)[1])
    push!(Z2_opt, -1 * objective_value(model; result = i)[2])
end

scatter(Z1_opt, Z2_opt,
    title        = L"$Y_N$ (algorithme $\epsilon$-constraint)",
    xlabel       = L"$z_1$ à maximiser",
    ylabel       = L"$z_2$ à minimiser",
    xlims        = (0, 30),
    ylims        = (0, 30),
    aspect_ratio = :equal,
    marker       = :circle,
    markersize   = 5,
    color        = :green,
    legend       = false
)

---
## Partie 4 — Exercice : Analyse discriminante et programmation linéaire

**Référence :** Ned Freed and Fred Glover.  
*A linear programming approach to the discriminant problem.*  
Decision Sciences, volume 12, issue 1, pages 68–74. 1981.

### Énoncé

**But :** Étant donné des observations appartenant à deux groupes **connus**, trouver une règle de classification permettant d'affecter une **nouvelle observation** à l'un des deux groupes.

**Exemple (Freed & Glover, 1981) :** 10 employés, 2 caractéristiques ($x_1$ = expérience, $x_2$ = formation).

| j | $x_1$ | $x_2$ | Groupe  |
|---|-------|-------|----------|
| 1 | 4 | 1 | Succès |
| 2 | 5 | 2 | Succès |
| 3 | 7 | 2 | Succès |
| 4 | 9 | 1 | Succès |
| 5 | 8 | 4 | Succès |
| 6 | 1 | 1 | Échec |
| 7 | 3 | 1 | Échec |
| 8 | 2 | 2 | Échec |
| 9 | 6 | 3 | Échec |
|10 | 3 | 3 | Échec |

### Approche : hyperplan séparateur

On cherche des poids $w_1, w_2$ et un seuil $c$ tels que :

- $\text{score}(j) = x_{1,j} \times w_1 + x_{2,j} \times w_2$
- $\text{score}(j) \geq c$ pour tout individu du groupe *Succès*
- $\text{score}(j) \leq c$ pour tout individu du groupe *Échec*

On introduit une **marge** $d$ et on maximise cette marge :

- Succès $\Rightarrow$ $\text{score}(j) \geq c + d$
- Échec $\Rightarrow$ $\text{score}(j) \leq c - d$

### Modèle LP

$$
\begin{array}{clrcrl}
\max & d \\\\
\text{s.t.} & x^s_{1,j} w_1 + x^s_{2,j} w_2 & - & d & \geq c & \quad \forall j \in \{1,\ldots,n_S\} \\
            & x^e_{1,j} w_1 + x^e_{2,j} w_2 & + & d & \leq c & \quad \forall j \in \{1,\ldots,n_E\} \\
            & w_1,\, w_2,\, d & \in & \mathbb{R} && \quad \text{(non contraint en signe)}
\end{array}
$$

### (a) Données de l'instance

In [ ]:
X_s = [ 4 1 ;   # une ligne = une observation
        5 2 ; 
        7 2 ; 
        9 1 ; 
        8 4  ]

X_e = [ 1 1 ; 
        3 1 ; 
        2 2 ; 
        6 3 ; 
        3 3  ]

c = 10
nS = size(X_s, 1)
nE = size(X_e, 1)

### (b) Modèle JuMP

In [ ]:
#using JuMP, GLPK

modAD = Model(GLPK.Optimizer)

@variable(modAD, w1)   
@variable(modAD, w2)
@variable(modAD, d)

@objective(modAD, Max, d)

@constraint(modAD, [j=1:nS], X_s[j,1] * w1 + X_s[j,2] * w2 - d ≥ c)
@constraint(modAD, [j=1:nE], X_e[j,1] * w1 + X_e[j,2] * w2 + d ≤ c)

print(modAD)

### (c) Résolution

In [ ]:
optimize!(modAD)

### (d) Extraction des résultats

In [ ]:
println("Statut : ", termination_status(modAD))

w1_opt = value(w1)
w2_opt = value(w2)
d_opt  = value(d)

println("w1 = ", round(w1_opt, digits=2),
        "  w2 = ", round(w2_opt, digits=2),
        "  d = ",  round(d_opt,  digits=2))

# Calcul des scores
scores_succes = X_s * [w1_opt; w2_opt]
scores_echecs = X_e * [w1_opt; w2_opt]

println("Scores Succès : ", round.(scores_succes, digits=1))
println("Scores Échec  : ", round.(scores_echecs,  digits=1))

### (e) Visualisation avec Plots.jl

In [ ]:
using Plots

# Calcul droite séparatrice et marges
x1r      = range(0, 12, length=200)
sep      = (c           .- w1_opt .* x1r) ./ w2_opt
marg_sup = (c + d_opt   .- w1_opt .* x1r) ./ w2_opt
marg_inf = (c - d_opt   .- w1_opt .* x1r) ./ w2_opt

# Affichage points
p = scatter(X_s[:,1], X_s[:,2],
        label="Succès", marker=:circle, ms=7, color=:darkgreen,
        title="Analyse discriminante par programmation linéaire",
        xlabel="x1", ylabel="x2")

scatter!(p, X_e[:,1], X_e[:,2],
        label="Échec", marker=:diamond, ms=7, color=:darkred)

# Affichage droites        
plot!(p, x1r, sep,      lw=2, color=:black, label="Seuil c")
plot!(p, x1r, marg_sup, lw=1, ls=:dash, color=:gray, label="Marge +d")
plot!(p, x1r, marg_inf, lw=1, ls=:dash, color=:gray, label="Marge -d")

display(p)